## Ensembles for Customer Satisfaction Prediction

KATE expects your code to define variables with specific names that correspond to certain things we are interested in.

KATE will run your notebook from top to bottom and check the latest value of those variables, so make sure you don't overwrite them.

* Remember to uncomment the line assigning the variable to your answer and don't change the variable or function names.
* Use copies of the original or previous DataFrames to make sure you do not overwrite them by mistake.

You will find instructions below about how to define each variable.

Once you're happy with your code, upload your notebook to KATE to check your feedback.

Businesses can improve their services by tailoring them to individual customers. One important factor is knowing when customers are dissatisfied. Based on their records, one can use machine learning tools to make predictions about which customers are more at risk of being dissatisfied than others. Such predictions allow for individualised actions that may help retain customers and will improve quality.

In this assignment, we will build a prediction model for bank account owners' satisfaction. The record includes more than 300 features for each client, including variables related to their balance and which banking operations they have performed. Many of these variables are sparse; some numerical, some categorical. 

Ensemble methods based on decision trees, such as random forests and boosting algorithms, have been very successful in modeling such heterogeneous tabular data. To learn how these models work, you will implement them step-by-step, and see how the performance of your predictions improve.

### Load the data

Load the data in `data/train_data.csv` with `pandas`. Inspect its content with `.head()`, `.shape` and other methods of your choice.

In [ ]:
import pandas as pd

df = pd.read_csv('data/train_data.csv')

In [ ]:
print(df.head())

print(df.shape)

df.info()



#### Target variable

The last column, named `TARGET`, is the variable to be predicted. `TARGET=1` represents a dissatisfied customer. Inspect the target column with `.value_counts()`. 

What is the proportion of dissatisfied customers? Is the dataset balanced or imbalanced?

In [ ]:
df['TARGET'].value_counts()

Separate the data into features `X` and target `y`. Split the data into `_train` and `_val` sets, using the default sklearn ratios and stratifying on `y` to keep the same level of imbalance.

*Hint: you may use `train_test_split()` for stratified splits.*

In [ ]:
from sklearn.model_selection import train_test_split


X = df.drop(columns=['TARGET', 'Unnamed: 0'], axis=1)
y = df['TARGET']

X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, random_state=1)



### Basic modelling pipeline

Implement a basic modelling pipeline for a Decision Tree Classifier, fitting the training data and printing the training and validation accuracy.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()

dtc.fit(X_train, y_train)

print(f'Decision Tree training accuracy: {dtc.score(X_train, y_train)}')
print(f'Decision Tree validation accuracy: {dtc.score(X_val, y_val)}')



Note that the accuracy score is quite high, even for this very simple model. Take a moment to think why this high score is not that significant.

#### ROC curve metric

Change your scoring metric to `roc_auc_score`, which calculates the area below the ROC curve of your **prediction probabilities**, instead of using the binary prediction decisions.

*Hint: `model.predict_proba(X)` will return the probabilities for `y = 0` and `y = 1`, slice this to only return the `y = 1` predictions. 

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred = dtc.predict_proba(X)[:, 1]

score = roc_auc_score(y, y_pred)

print(f"roc Score: {score:.4f}")

#### Baseline score for random predictions

Calculate the ROC AUC for random uniform prediction probabilities. 

Is the Decision Tree better? Based on the training and validation scores, what is the problem with the Decision Tree model?

*Hint: You can use `np.random.uniform` to create an array of random numbers under 1.*

In [ ]:
import numpy as np

randoms = np.random.uniform(low=0, high=1, size=len(y_val))

random_auc = roc_auc_score(y_val, randoms)

print(f"Random ROC AUC: {random_auc:.4f}")


Create a function named `test_model(model, X_train, y_train, X_val, y_val)` that performs the basic prediction pipeline, receiving as argument the model and data, fitting the training data, and returning the training and validation `roc_auc_score`. Check that it works with the Decision Tree model.

In [ ]:
def test_model(model, X_train, y_train, X_val, y_val):
    
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, y_pred)
    
    return auc_score

score = test_model(dtc, X_train, y_train, X_val, y_val)

print(f"ROC AUC: {score:.4f}")

## Optimising decision trees 

Optimise your model by changing the `max_depth` hyperparameter. Iterate through and calculate the training and validation scores for depths of `[1,2,5,10,20,50]`. Which depth has the best performance? Remember that we want a model that performs well on both the training and validation datasets. 

In [ ]:
depths = [1,2,5,10,20,50]

for d in depths:
    dtc = DecisionTreeClassifier(max_depth=d)

    dtc.fit(X_train, y_train)

    print(f'Decision Tree training accuracy: \t depth {d} \t {dtc.score(X_train, y_train)}')
    print(f'Decision Tree validation accuracy: \t depth {d} \t {dtc.score(X_val, y_val)}')

To evaluate your models, we will test your data on an unseen testing set. Load the test at `data/test_data.csv`.

In [ ]:
test_data = pd.read_csv('data/test_data.csv')


 Calculate the prediction probabilities for the `test_data` using the best Decision Tree depth, saving them in a variable named `dtc_preds`. `dtc_preds` should be an numpy array containing the probabilities for `y = 1` only.

In [ ]:
max_depth = 2

dtc = DecisionTreeClassifier(max_depth=max_depth)

#if 'Unnamed: 0' in X_train.columns :
#X_train.drop(columns=['Unnamed: 0'], axis=1, inplace=True)
dtc.fit(X_train, y_train)

dtc_preds = dtc.predict_proba(test_data)[:, 1]

print(dtc_preds)


### Bagging

While decision trees are prone to overfitting, their ensemble can be a powerful predictor. A bagging ensemble of decision trees uses the average prediction of multiple decision trees base models, that have each been trained with on a different set of data samples. This resampling processes is known as Bootstraping. (The name Bagging comes from Bootstrap + Aggregation, meaning bootstrap your data and then aggregate your model outputs.)

You will create a Bagging model class, named `myBagging`, filling in the class structure below.

The `.fit()` method should fit each base model with a bootstrap sample of the data (with replacement), with data size proportional by the hyperparameter `subsample`. That is, if `subsample=0.5`, each base model should get half the total number of samples.

The `.predict_proba()` method should estimate and average the prediction probabilities of the base models.

*Hint: You can use the `resample()` function for creating bootstrap samples.*

In [ ]:
class myBagging:
    def __init__(self, base_models, subsample = 1.):
        self.n_models = len(base_models)
        self.base_models = base_models
        self.subsample = subsample
        
    def fit(self, X, y):
        '''Loop over base models, generate a bootstrap sample of the data with 'resample()',
           and fit them to the data.
           
           To access the variables inside the myBagging class, use the 'self.' prefix, 
           i.e. self.base_models, self.n_models and self.subsample
        '''
        from sklearn.utils import resample

        self.fitted_models_ = []
        for base_model in self.base_models:
            num_samples = int(len(X) * self.subsample)
            X_res, y_res = resample(X, y, replace=True, n_samples=num_samples, random_state=1)
            model = base_model 
            model.fit(X_res, y_res)
            self.fitted_models_.append(model)
        return self

    def predict_proba(self, X):
        '''Return the ensemble predictions, given by the average prediction probability over base models.
           It should be an array with the length of the dataset.'''

        import numpy as np
        preds = np.array([model.predict_proba(X) for model in self.fitted_models_])
        avg_preds = preds.mean(axis=0)
        return avg_preds
    
    def score(self, X, y):
        from sklearn.metrics import roc_auc_score
        y_pred = self.predict_proba(X)
        return roc_auc_score(y, y_pred)
    


Run and score a Bagging ensemble, with 10 decision trees as base learners, with maximum depth 10 and subsample 0.5. Use your `myBagging` class and `test_model()`.

In [ ]:
# Your code here...
num_trees = 10
max_depth = 10
sub_sample = 0.5

dtcs = [DecisionTreeClassifier(max_depth=max_depth) for _ in range(num_trees)]

mb = myBagging(base_models=dtcs, subsample=sub_sample)
mb.fit(X=X_train, y=y_train)

score = test_model(mb, X_train, y_train, X_val, y_val)
print("Validation ROC AUC:", score)

mb_test_preds = mb.predict_proba(test_data)
print(mb_test_preds.shape)


### Sklearn comparison

For comparison, run the below cell to see how the `sklearn` implementation, `BaggingClassifier` does. 

Is there much of a difference? Why could that be?

In [ ]:
from sklearn.ensemble import BaggingClassifier

test_model(BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=10), 
                             n_estimators=10, max_samples=0.5), X_train, y_train, X_val, y_val)

### Random Forests

Random Forests are bagged ensembles of decision trees in which, at each node split during training, only a fraction of the features are considered for the optimal split (e.g. for optimal Gini gain).

Run and score a Random Forest version of your `myBagging` classifier, by changing the `max_features` parameter inside the base learner.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

max_features = [1, 2, 3, 5, 10]
sub_sample = 0.5

for mf in max_features: 
    rfc = [RandomForestClassifier(max_features=mf)]

    mb = myBagging(base_models=rfc, subsample=sub_sample)
    mb.fit(X=X_train, y=y_train)

    score = test_model(mb, X_train, y_train, X_val, y_val)
    print("Validation ROC AUC:", score)

    mb_test_preds = mb.predict_proba(test_data)
    print(mb_test_preds.shape)

### Sklearn comparison

For comparison, run and score the `sklearn` implementation, `RandomForestClassifier`.

Is there much of a difference? Why could that be?

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=5)

# Your code below

rf.fit(X_train, y_train)

rf_y_pred = rf.predict_proba(X_val)[:, 1]

auc_score = roc_auc_score(y_val, rf_y_pred)

print(f'RF score: {auc_score}')


### Optimise your Random Forest

Optimise your Random Forest hyperparameters, both for the myBagging class and for the base decision trees. 

Calculate the prediction probabilities for the `test_data` using your best random forest, saving them in a variable named `rf_preds`. Like before `rf_preds` should be an numpy array containing the probabilities for `y = 1` only.


In [ ]:
# TODO Gridsearch
from sklearn.model_selection import GridSearchCV

params = dict(
    max_features = [1, 2, 3, 5, 10],
    max_depth    = [5, 10, 15, 20]
)

rfc = RandomForestClassifier(n_estimators=100, random_state=42)

gcv = GridSearchCV(estimator = rfc,
                    param_grid=params,
                    n_jobs=-1,
                    verbose=1)

gcv.fit(X_train, y_train)

score = test_model(gcv, X_train, y_train, X_val, y_val)
print("Validation ROC AUC:", score)

# print("Running MyBagging")


# max_score = 0

# for f in params["max_features"]:
#     for d in params["max_depth"]:
#         rfc = [RandomForestClassifier(max_features=f, max_depth=d) for _ in range(20)]
#         mb = myBagging(base_models=rfc, subsample=0.5)
#         mb.fit(X=X_train, y=y_train)

#         score = test_model(mb, X_train, y_train, X_val, y_val)
#         if score > max_score:
#             max_score = score
#             print("Max Features:", f)
#             print("Max Depth:", d)
#             print("Validation ROC AUC:", score)

rf_preds = mb.predict_proba(test_data)[:, 1]


print(type(rf_preds))



###### EXISTING #######
# rfc = [RandomForestClassifier(max_features=mf, max_depth=5)]

# mb = myBagging(base_models=rfc, subsample=sub_sample)
# mb.fit(X=X_train, y=y_train)

# score = test_model(mb, X_train, y_train, X_val, y_val)
# print("Validation ROC AUC:", score)

# rf_preds = mb.predict_proba(test_data)[:, 1]

# print(type(rf_preds))

In [ ]:
visitors = {'user123', 'user456'}
visitors.add('xx')
visitors

Note that including more decision trees improve performance but increases the computational cost of training linearly. The `max_depth` and `max_features` arguments can heavily cut the training time, by reducing the tree size and number of features considered at each split.

## Gradient Boosting

We will now implement a more sophisticated ensemble, Gradient Boosting, in which the base models are trained sequentially. Each new base model predicts what previous base models missed. 

As gradient boosting requires a continuous gradient, it can only use regression models for the base learner. 

For this exercise, we will perform regression directly on the 0-1 class labels, and treat the raw outputs as probabilities. 

We will try to setup the base models to optimise the MSE loss function against the class-labels, for which the gradient becomes simply the residual errors. 

When applied to probabilities, the MSE is known as the Brier score. 

Whilst performing this exercise, have a think about whether this is a robust approach. 

If not, what would you change either to your base-learners, meta-algorithm, or evaluation metrics to make this more robust?

In the below structure, fill the `.fit()` and `.predict_proba()` functions. When returning probabilities remember that you will need to return an array containing the probabilities for `y = 0` and `y = 1`. 

In [ ]:
class myGradientBoosting:
    
    def __init__(self, base_models, learning_rate=0.5):
        self.n_models = len(base_models)
        self.models = base_models
        self.learning_rate = learning_rate
    
    def fit(self, x, y):
        ''' The `.fit()` function should loop over each base model 
         fitting it to the residual of the ensemble predictions so far, for the MSE loss:
         
         predictions = 0
         for each base model:
             residual = y - predictions   
             fit base model and make new predictions
             predictions = predictions + learning_rate * new_prediction 
        '''
        predictions = 0
        self.fitted_models_ = []
        for base_model in self.models:
            residual = y - predictions   

            model = base_model.fit(x,residual)
            new_prediction = model.predict(x)
          
            predictions = predictions + self.learning_rate * new_prediction
            self.fitted_models_.append(model)
       
    def predict_proba(self, x):
        ''' Generate the ensemble prediction, by looping over each base model.
            Get their predictions and sum them, scaled by the learning rate.
        
            Trick: Regressor models return only one prediction (instead of two probabilities in the Classifiers).
                   To make your class compatible with test_model(), you need to output the probabilities for both
                   y = 0 and y = 1, this can be done using np.array([1-probs, probs]).T'''
        import numpy as np
        probs_sum = np.zeros(len(x))
        for model in self.fitted_models_:
            pred = model.predict(x)
            print(pred)
            probs_sum += self.learning_rate * pred
        probs_sum = np.clip(probs_sum, 0, 1)
        return np.array([1 - probs_sum, probs_sum]).T



Run and score a Gradient Boosting model, with 20 regression decision trees as your base learners, with maximum depth 5, maximum feature 0.5, and learning rate 0.5. Use your `myGradientBoosting` class and `test_model()`. 

In [ ]:
from sklearn.tree import DecisionTreeRegressor

num_trees = 20
max_depth = 5
maximum_feature = 0.5
learning_rate = 0.5

dtrs = [DecisionTreeRegressor(max_features=maximum_feature, max_depth=max_depth) for _ in range(num_trees)]

mgb = myGradientBoosting(base_models=dtrs, learning_rate=learning_rate)
mgb.fit(x=X_train,y=y_train)

score = test_model(mgb, X_train, y_train, X_val, y_val)

print("Validation ROC AUC:", score)

mgb_test_preds = mgb.predict_proba(test_data)
print(mgb_test_preds.shape)

### Sklearn comparison

For comparison, run and score the `sklearn` implementation, `GradientBoostingClassifier` in the cell below.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

test_model(GradientBoostingClassifier(max_features=0.5, max_depth=5), X_train, y_train, X_val, y_val)

### Optimise your Gradient Boosting 

Optimise your myGradientBoosting and decision tree hyperparameters.

Calculate the prediction probabilities for the `test_data` using your best gradient boosting model saving them in a variable named `gb_preds`. Like before `gb_preds` should be an numpy array containing the probabilities for `y = 1` only.


In [ ]:
## Gridsearch


gbc = [GradientBoostingClassifier(max_features=0.7, max_depth=5)]

mgb = myGradientBoosting(base_models=gbc, learning_rate=0.5)
mgb.fit(x=X_train, y=y_train)

score = test_model(mgb, X_train, y_train, X_val, y_val)
print("Validation ROC AUC:", score)

gb_preds = mgb.predict_proba(test_data)[:, 1]

Try to think about the difference between your implementation and the GradientBoostingClassifier.

Are there any fundamental differences? If so, why?

You could try looking at the distribution of your output probabilities for each model.